# Procesamiento de datos - Capa Gold (Latinoamérica)

Este notebook construye la capa Gold a partir del consolidado Silver en Delta Lake.


In [1]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip

active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

builder = (
    SparkSession.builder
    .appName("gold-worldbank-latam")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
SILVER_PATH = PROJECT_ROOT / "data" / "silver" / "economic_indicators"
GOLD_PATH = PROJECT_ROOT / "data" / "gold" / "economic_indicators_latam"

(PROJECT_ROOT / "data" / "gold").mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Silver path: {SILVER_PATH}")
print(f"Gold path: {GOLD_PATH}")

Project root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation
Silver path: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\silver\economic_indicators
Gold path: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\gold\economic_indicators_latam


In [5]:
LATAM_CODES = {
    "ARG", "BOL", "BRA", "CHL", "COL", "CRI", "CUB", "DOM", "ECU", "SLV",
    "GTM", "HTI", "HND", "MEX", "NIC", "PAN", "PRY", "PER", "URY", "VEN",
    "BLZ", "GUY", "SUR", "JAM", "TTO", "BHS", "BRB", "ATG", "DMA", "GRD",
    "KNA", "LCA", "VCT",
}

silver_df = spark.read.format("delta").load(str(SILVER_PATH))

required_base_cols = ["country_code", "country_name", "year"]
indicator_cols = [c for c in silver_df.columns if c not in required_base_cols]

gold_latam = (
    silver_df
    .filter(F.col("country_code").isin(*sorted(LATAM_CODES)))
    .withColumn("year", F.col("year").cast("int"))
    .select(*required_base_cols, *indicator_cols)
    .dropDuplicates(["country_code", "year"])
)

if {"exports", "imports"}.issubset(set(gold_latam.columns)):
    gold_latam = gold_latam.withColumn("trade_balance", F.col("exports") - F.col("imports"))

(
    gold_latam.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(GOLD_PATH))
)

print(f"[OK] Gold LatAm guardado en Delta Lake: {GOLD_PATH}")
print(f"[OK] Filas Gold LatAm: {gold_latam.count()}")

[OK] Gold LatAm guardado en Delta Lake: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\gold\economic_indicators_latam
[OK] Filas Gold LatAm: 2055
